# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrabansal10/FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/rudrabansal10/FlyRank_Internship/43b468d73eba109085f02d01f3a59754d5356453/data/raw/content_refresh_anonymized.csv")


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule
A page is a stronger review candidate when it is

- visible,

- ranks in positions 1–20, and has CTR below 0.5%.

These are based on the signal audit we performed before.

The rule identifies content that is most likely to benefit from a refresh.

The score is intended as a decision-support ranking rather than a prediction.

The rule can produce reason codes -

- "LOW_CTR_VISIBLE_PAGE"
- "HIGH_SEARCH_VISIBILITY"
- "GENERAL_REVIEW"

Action Label:
- Refresh Content


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import numpy as np

# A page is a stronger review candidate when it is visible,
# ranks in positions 1–20, and has CTR below 0.5%.
df["low_ctr_visible"] = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).astype(int)

# Visibility gives higher-impact pages a modest priority boost.
df["visibility_score"] = df["impressions_90d"].rank(pct=True)

# Simple fixed baseline: 70% supported low-CTR signal, 30% visibility.
df["baseline_score"] = (
    0.70 * df["low_ctr_visible"]
    + 0.30 * df["visibility_score"]
)

def reason_code(row):
    if row["low_ctr_visible"] == 1:
        return "LOW_CTR_VISIBLE_PAGE"
    if row["impressions_90d"] >= 500:
        return "HIGH_SEARCH_VISIBILITY"
    return "GENERAL_REVIEW"

df["reason_code"] = df.apply(reason_code, axis=1)

df["action"] = np.where(
    df["low_ctr_visible"] == 1,
    "Review title, metadata, and search snippet",
    "Monitor or review when capacity allows",
)

high_cutoff = df["baseline_score"].quantile(0.80)
medium_cutoff = df["baseline_score"].quantile(0.50)

def confidence(score):
    if score >= high_cutoff:
        return "High"
    if score >= medium_cutoff:
        return "Medium"
    return "Low"

df["confidence"] = df["baseline_score"].apply(confidence)

queue = df.sort_values(
    ["baseline_score", "impressions_90d"], ascending=False,).reset_index(drop=True)

queue["rank"] = queue.index + 1

display(queue[[
            "rank", "reason_code", "action", "confidence",
            "impressions_90d", "ctr", "avg_position",
            "baseline_score"]].head(20))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

index,rank,reason_code,action,confidence,impressions_90d,ctr,avg_position,baseline_score

0,1,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,517715,0.14,4.2,1.0

1,2,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,517109,0.25,5.4,0.9999899999999999

2,3,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,509252,0.15,2.5,0.9999799999999999

3,4,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,463103,0.41,2.3,0.99996

4,5,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,416180,0.23,4.0,0.9999399999999999

5,6,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,345111,0.21,5.4,0.9999199999999999

6,7,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,309910,0.16,5.6,0.9999

7,8,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,295097,0.05,7.3,0.9998799999999999

8,9,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,272144,0.03,2.3,0.99984

9,10,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,236803,0.26,4.4,0.9998199999999999

10,11,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,223271,0.03,7.8,0.99979

11,12,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,213963,0.1,4.7,0.9997499999999999

12,13,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,211366,0.41,5.1,0.99974

13,14,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,208798,0.23,5.2,0.99973

14,15,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,208678,0.0,9.7,0.9997199999999999

15,16,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,201584,0.24,5.8,0.9996999999999999

16,17,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,201111,0.11,5.7,0.99969

17,18,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,
198671,0.18,5.6,0.9996799999999999

18,19,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,
197199,0.22,6.8,0.99967

19,20,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",High,
192478,0.29,5.7,0.99966

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some recommendations may be incorrect because:

- Older pages are not always outdated.
- Low CTR can result from highly competitive search results rather than poor content.
- Pages with low engagement may intentionally answer simple questions quickly.
- Seasonal topics naturally fluctuate in performance.
- Some pages may have been recently refreshed, but the historical metrics have not yet reflected the changes.

Leakage check:

✔ trend_direction was not used.

✔ trend_pct was not used.

✔ is_declining_label was not used.

✔ No future-window metrics were used.

✔ Only historical content and performance metrics available at the decision time were used.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

def precision_at_k(labels, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

_, test_index = next(
    splitter.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"],
    )
)

test_df = df.iloc[test_index]

print("Base rate:", round(test_df["is_declining_label"].mean(), 3))
print(
    "Baseline Precision@50:",
    round(
        precision_at_k(
            test_df["is_declining_label"],
            test_df["baseline_score"],
            50,
        ),
        3,
    ),
)

Base rate: 0.511
Baseline Precision@50: 0.42


In [ ]:
import json

metrics = {
    "top_20_size": 20,
    "rule_name": "baseline_action_score",
    "num_candidates": len(queue),
    "mean_score": float(queue["baseline_score"].mean()),
    "max_score": float(queue["baseline_score"].max()),
    "heldout_base_rate": 0.511,
    "baseline_precision_at_50": 0.420
}

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)